# 00 LocalEMA Single-Client Training

This notebook runs the LocalEMA comparison under the same DQA-matched single-client EfficientTeacher protocol. The only method change versus `01_efficient_teacher_training.ipynb` is that each client's previous EMA teacher is carried into the next round.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "efficient_teacher/run_efficient_teacher_single_client.py",
        "navigating_data_heterogeneity/setup_fedsto_exact_reproduction.py",
        "navigating_data_heterogeneity/vendor/efficientteacher/train.py",
    )
    candidates = []
    for base in (start, *start.parents):
        candidates.extend([base, base / "Object_Detection"])
    for candidate in candidates:
        if all((candidate / marker).exists() for marker in required):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Object_Detection repository root.")


REPO_ROOT = find_repo_root()
ET_ROOT = REPO_ROOT / "efficient_teacher"
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = ET_ROOT / "run_efficient_teacher_single_client.py"
WORK_ROOT = ET_ROOT / "efficientteacher_localema"
PSEUDO_STATS_ROOT = WORK_ROOT / "pseudo_stats"
DQA_WARMUP_CHECKPOINT = (
    DQA_ROOT
    / "efficientteacher_dqa_cwa_corrected_12h"
    / "global_checkpoints"
    / "round000_warmup.pt"
)
PYTHON_BIN = Path(sys.executable)
TRAIN_LOG = WORK_ROOT / "efficientteacher_latest.log"

pd.options.display.max_columns = 200
print("repo:", REPO_ROOT)
print("workspace:", WORK_ROOT)

## 1. DQA-Matched Configuration

The defaults mirror the current guarded DQA notebook. Change `CLIENT_ID` only if you want a different single weather client.

In [ ]:
CLIENT_ID = 0
CLIENT_WEATHER = None
WARMUP_EPOCHS = 15
PHASE1_ROUNDS = 14
PHASE2_ROUNDS = 27
BATCH_SIZE = 64
WORKERS = 0
REQUESTED_GPUS = 2
MASTER_PORT = 29521
MIN_FREE_GIB = 70
LOCAL_EMA = True
USE_DQA_WARMUP_CHECKPOINT = True
WARMUP_CHECKPOINT = DQA_WARMUP_CHECKPOINT if USE_DQA_WARMUP_CHECKPOINT else None
APPEND_TRAIN_LOG = False
EXTRA_RUN_ARGS = []

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), but {AVAILABLE_CUDA_GPUS} visible; using {GPUS}.")

config_summary = {
    "client_id": CLIENT_ID,
    "client_weather": CLIENT_WEATHER or "from manifest",
    "warmup_epochs": WARMUP_EPOCHS,
    "phase1_rounds": PHASE1_ROUNDS,
    "phase2_rounds": PHASE2_ROUNDS,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "gpus": GPUS,
    "local_ema": LOCAL_EMA,
    "warmup_checkpoint": str(WARMUP_CHECKPOINT) if WARMUP_CHECKPOINT else None,
    "workspace": str(WORK_ROOT),
}
config_summary

## 2. Build Data Lists and Configs

This uses the same `setup_fedsto_exact_reproduction.py` path as DQA/FedSTO, but patches the client list down to one selected client inside the runner.

In [ ]:
setup_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--setup-only",
    "--workspace-root",
    str(WORK_ROOT),
    "--pseudo-stats-root",
    str(PSEUDO_STATS_ROOT),
    "--client-id",
    str(CLIENT_ID),
]
if CLIENT_WEATHER:
    setup_cmd.extend(["--client-weather", CLIENT_WEATHER])

subprocess.run(setup_cmd, cwd=REPO_ROOT, check=True)

manifest_path = WORK_ROOT / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_summary = {
    "classes": manifest["classes"],
    "server_train_images": manifest["server"]["train_images"],
    "server_val_images": manifest["server"]["val_images"],
    "clients": manifest["clients"],
    "paper_schedule": manifest["paper_schedule"],
}
manifest_summary

## 3. Dependency and Dry-Run Check

The dry run generates runtime commands and YAMLs without launching training.

In [ ]:
import importlib.util

required = {
    "yaml": "PyYAML",
    "cv2": "opencv-python",
    "thop": "thop",
    "tensorboard": "tensorboard",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError("Missing runtime dependencies: " + ", ".join(missing))

dry_run_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--dry-run",
    "--workspace-root",
    str(WORK_ROOT),
    "--pseudo-stats-root",
    str(PSEUDO_STATS_ROOT),
    "--client-id",
    str(CLIENT_ID),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
]
if CLIENT_WEATHER:
    dry_run_cmd.extend(["--client-weather", CLIENT_WEATHER])
dry_run_cmd.append("--local-ema" if LOCAL_EMA else "--no-local-ema")
if WARMUP_CHECKPOINT is not None:
    dry_run_cmd.extend(["--warmup-checkpoint", str(WARMUP_CHECKPOINT)])

subprocess.run(dry_run_cmd, cwd=REPO_ROOT, check=True)

## 4. Start or Resume Training

This cell is restartable. The runner reuses valid checkpoints, appends compact status to notebook output, and writes full EfficientTeacher logs to `efficientteacher_latest.log`.

When `WARMUP_CHECKPOINT` points at the DQA warm-up checkpoint, the run skips raw-pretrained warm-up training and seeds `round000_warmup.pt` from that DQA artifact instead. If this workspace is already running with a raw-pretrained warm-up, stop that old process before rerunning this cell.

In [ ]:
RUN_FULL_REPRODUCTION = True
ALLOW_CPU_TRAINING = False
FORCE_RESTART = False
FORCE_WARMUP = False
FORCE_RETRAIN = False

train_cmd = [
    str(PYTHON_BIN),
    "-u",
    str(RUN_SCRIPT),
    "--workspace-root",
    str(WORK_ROOT),
    "--pseudo-stats-root",
    str(PSEUDO_STATS_ROOT),
    "--client-id",
    str(CLIENT_ID),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
]
if CLIENT_WEATHER:
    train_cmd.extend(["--client-weather", CLIENT_WEATHER])
train_cmd.append("--local-ema" if LOCAL_EMA else "--no-local-ema")
if WARMUP_CHECKPOINT is not None:
    train_cmd.extend(["--warmup-checkpoint", str(WARMUP_CHECKPOINT)])
if APPEND_TRAIN_LOG:
    train_cmd.append("--append-train-log")
if FORCE_RESTART:
    train_cmd.append("--force-restart")
if FORCE_WARMUP:
    train_cmd.append("--force-warmup")
if FORCE_RETRAIN:
    train_cmd.append("--force-retrain")
train_cmd.extend(EXTRA_RUN_ARGS)

if RUN_FULL_REPRODUCTION and AVAILABLE_CUDA_GPUS < 1 and not ALLOW_CPU_TRAINING:
    print("No CUDA GPU is visible, so the run was not started.")
elif RUN_FULL_REPRODUCTION:
    print("starting:", " ".join(train_cmd))
    subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
else:
    print("Set RUN_FULL_REPRODUCTION = True to launch or resume training.")

print("train log:", TRAIN_LOG)

## 5. Progress Snapshot

In [ ]:
def modified_utc(path: Path) -> str:
    if not path.exists():
        return ""
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat()


history_path = WORK_ROOT / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
completed_phase1 = sum(1 for entry in history if int(entry.get("phase", 0)) == 1)
completed_phase2 = sum(1 for entry in history if int(entry.get("phase", 0)) == 2)
display(
    pd.DataFrame(
        [
            {"phase": "phase1", "completed": completed_phase1, "target": PHASE1_ROUNDS},
            {"phase": "phase2", "completed": completed_phase2, "target": PHASE2_ROUNDS},
        ]
    )
)

artifacts = [
    WORK_ROOT / "manifest.json",
    WORK_ROOT / "history.json",
    WORK_ROOT / "global_checkpoints" / "round000_warmup.pt",
    WORK_ROOT / "global_checkpoints" / f"phase2_round{PHASE2_ROUNDS:03d}_global.pt",
    TRAIN_LOG,
    PSEUDO_STATS_ROOT,
]
display(
    pd.DataFrame(
        [
            {
                "path": str(path),
                "exists": path.exists(),
                "bytes": path.stat().st_size if path.exists() and path.is_file() else "",
                "modified_utc": modified_utc(path),
            }
            for path in artifacts
        ]
    )
)
display(pd.DataFrame(history).tail(10) if history else pd.DataFrame())

## 6. Recent Logs

In [ ]:
def tail(path: Path, lines: int = 80) -> str:
    if not path.exists():
        return f"{path} does not exist yet."
    result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True)
    return result.stdout


print(tail(TRAIN_LOG, 80))